# 01. Comprensión de los Datos — Microdatos ENDES INEI Perú 2023

**Universidad Nacional Federico Villarreal (UNFV)**  
**Facultad de Ingeniería Electrónica e Informática (FIEI)**  
**Autor:** Jimenez Villalva Franz José  
**Asignatura:** Inteligencia Artificial / Metodología de la Investigación Científica  

Este primer cuaderno corresponde a la fase de **Data Understanding** según la metodología CRISP-DM. Aquí revisamos los datos brutos del INEI tal como vienen de la encuesta, para entender cómo están organizados los archivos y qué significa cada columna antes de meterles mano en la limpieza.

## 1. Inspección de Microdatos Crudos del INEI (ENDES 2023)
Trabajamos con dos tablas oficiales que luego juntaremos por el código del hogar:
- **`RECH6_2023.csv`**: Datos de salud de los niños (peso, talla y la prueba de hemoglobina).
- **`RECH0_2023.csv`**: Datos de la vivienda (ubicación, departamento y la altitud en msnm).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_R6 = ROOT_DIR / 'data' / 'raw' / 'RECH6_2023.csv'
RAW_R0 = ROOT_DIR / 'data' / 'raw' / 'RECH0_2023.csv'

print(':: Leyendo los microdatos crudos del INEI...')
r6 = pd.read_csv(RAW_R6, low_memory=False)
r0 = pd.read_csv(RAW_R0, low_memory=False)

# Limpiamos espacios y caracteres raros en los nombres de columnas
r6.columns = [c.strip().replace('ï»¿', '') for c in r6.columns]
r0.columns = [c.strip().replace('ï»¿', '') for c in r0.columns]

print(f'Módulo RECH6 (Niños): {r6.shape[0]} filas x {r6.shape[1]} columnas')
print(f'Módulo RECH0 (Hogar): {r0.shape[0]} filas x {r0.shape[1]} columnas')

## 2. Diccionario de Variables Crudas Clave

| Módulo | Variable | Descripción | Unidad / Significado |
| :--- | :--- | :--- | :--- |
| **RECH6** | `HHID` | Identificador del hogar | Código para hacer el match entre tablas |
| **RECH6** | `HC1` | Edad del menor | Meses cumplidos (de 0 a 59) |
| **RECH6** | `HC27` | Sexo | 1: Masculino, 2: Femenino |
| **RECH6** | `HC53` | Hemoglobina medida en sangre | Viene multiplicada por 10 (ej. 110 es 11.0 g/dL) |
| **RECH6** | `HC55` | Resultado de la prueba | 0: Medición exitosa, otro valor: No se pudo medir |
| **RECH6** | `HC2` | Peso corporal | En hectogramos (dividido entre 10 da kg) |
| **RECH6** | `HC3` | Talla o estatura | En milímetros (dividido entre 10 da cm) |
| **RECH0** | `HHID` | Identificador del hogar | Llave de unión con RECH6 |
| **RECH0** | `HV040` | Altitud geográfica | Metros sobre el nivel del mar (msnm) |
| **RECH0** | `HV024` | Departamento | Código del 1 al 25 |
| **RECH0** | `HV025` | Tipo de zona | 1: Urbano, 2: Rural |

In [ ]:
# Vemos cómo lucen las primeras filas de ambas tablas
print('--- Primeras filas de RECH6 (Datos de niños) ---')
display(r6[['HHID', 'HC1', 'HC27', 'HC53', 'HC55', 'HC2', 'HC3']].head())

print('\n--- Primeras filas de RECH0 (Vivienda y Altitud) ---')
display(r0[['HHID', 'HV040', 'HV024', 'HV025']].head())

## 3. Demostración del Cruce por Hogar (`HHID`)
Al hacer match con el código `HHID`, cada niño queda vinculado con la altitud exacta del lugar donde vive, lo cual es fundamental para calcular luego la corrección del MINSA.

In [ ]:
r6['HHID'] = r6['HHID'].astype(str).str.strip()
r0['HHID'] = r0['HHID'].astype(str).str.strip()

merged_sample = pd.merge(
    r6[['HHID', 'HC1', 'HC53', 'HC55']],
    r0[['HHID', 'HV040', 'HV024', 'HV025']],
    on='HHID',
    how='inner'
)
print(f'Total de registros que cruzaron correctamente: {merged_sample.shape[0]}')
display(merged_sample.head())

## 4. Revisión del Dataset Final Procesado
Comprobamos el archivo final limpio (`anemia_infantil_peru_endes.csv`) que utilizaremos en el modelado, confirmando que no tenga valores nulos.

In [ ]:
DATA_PATH = ROOT_DIR / 'data' / 'processed' / 'anemia_infantil_peru_endes.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset procesado listo: {df.shape[0]} niños y {df.shape[1]} columnas')

print('\nComprobando si quedó algún valor nulo:')
print(df.isnull().sum())

print('\nResumen estadístico de las variables:')
display(df.describe().T)